# 19 · An ensemble on the p scale, and whether its verdicts are stable (Phase 5)

**Goal.** Build the final system from the **original recipe** (the owner's decision after the 13-fold evidence): three seeds, combined on the val-human tail-probability ("p") scale of notebook 18, with the threshold rule that the declared Phase 4 choice rule picked for the original recipe (rule (d), notebook 18b). Then check (1) that the Phase 4 threshold still means what it should for an ensemble, and (2) that the ensemble's verdicts are more stable than a single model's. The notebook ends with the **exact table of the single final held-out check**, which is defined here and **not computed** (the project owner sees its definition first).

**Everything in sections 1-7 was written and committed to git before any Phase 5 analysis was run.** The GPU jobs that produce the data (two more seeds of the original recipe on the full training set; leave-one-prompt-out fold 11 with three seeds) are prerequisites, not analyses, and were started first. The held-out prompts and held-out generators are not scored anywhere in this notebook before section 7 is signed off.

## 1 · The three models and where their scores come from

| seed | model | val and test scores |
|---|---|---|
| 42 | `models/distilroberta_finetuned` (notebook 08, the existing model; first 512 tokens) | `results/15_scores.csv`, column `roberta`, **only the val and test rows are used** (that file also holds held-out rows from Phase 1; they are not read) |
| 43 | `models/distilroberta_orig_s43` (same recipe, trained for this phase on all 28,006 training essays) | `results/17_final/orig_s43_scores.csv` (val and test only) |
| 44 | `models/distilroberta_orig_s44` (same) | `results/17_final/orig_s44_scores.csv` (val and test only) |

The recipe (notebook 08): DistilRoBERTa, 2 epochs, lr 2e-5, 6% warm-up, 512 tokens, fp16, batch 16, best epoch by val AUC. No augmentation, no weights.

## 2 · The p scale per seed, the ensemble, and the combination rule (chosen on val by a rule declared here)

- **Per seed:** $p_s(x) = (1 + \#\{\text{val humans with score} \ge x\}) / (1 + n)$ with the 2,291 val humans (all 13 ordinary prompts), exactly as in notebook 18. Small = more AI-like.
- **Ensemble score:** $e(x) = \text{combine}(p_{42}(x), p_{43}(x), p_{44}(x))$ with *combine* either the **mean** or the **median** of the three p-values.
- **Ensemble p:** $e$ is not itself a p-value, so it is re-calibrated the same way: $p_{\text{ens}}(x) = (1 + \#\{\text{val humans } h : e(h) \le e(x)\}) / (1 + n)$. All thresholds $\tau$ below are on this scale, so the three seeds, the ensemble and any single model are on one scale.
- **Choice of the combination (val only, rule declared before computing):** the combination with the higher share of **val AI essays with $p_{\text{ens}} \le 0.01$** (val AUC and the share caught at the 1% level are the only val statistics used; the val AI essays are not used for calibration, so this is not circular). If the two differ by less than **0.2 percentage points** (about two of the 1,210 val AI essays), the **mean** is chosen (declared tie-break: the simpler one). `test` and the held-out sets play no part in this choice.

## 3 · Testing the Phase 4 assumption: does the threshold carry over from single models to the ensemble?

Phase 4's thresholds were derived from **single-seed** leave-one-prompt-out models. The assumption that carries them to the ensemble (notebook 18, section 7): on a new topic, human essays land in the low-$p$ tail zone **at the same rates for the ensemble as for a single model**. Testable on two folds with three seeds of the original recipe:

- **Fold 7 (Facial action coding system)**: seeds 42, 43, 44 from notebook 16b (`results/16b_lopo`).
- **Fold 11 (Seeking multiple opinions)**: seeds 42, 43, 44 from this phase (`results/19_phase5/lopo`), chosen *by a rule declared now*: the prompt with the highest false-flag rate for the original recipe (5.80% in 16b) that does not already have three seeds, i.e. where an inflation effect is largest and the test is most informative. (Fold 11 is also the direct check on regression to the mean: its original-recipe rate with other seeds.)

For each fold, with each seed's p of the left-out humans computed against *that seed's own* other-prompt val humans (the same essays for every seed), and the ensemble's $p_{\text{ens}}$ computed from the seeds' p-values with the combination chosen in section 2 and calibrated on the same other-prompt val humans:

- the **false-flag rate at $p \le \tau$** for $\tau = 0.01$ and $0.005$ (nominal rates), for each single seed and for the ensemble, with Wilson intervals, and the **inflation factor** = observed rate / $\tau$;
- the **paired difference** (ensemble minus the average of the three singles) in false-flag rate, with a cluster bootstrap interval (1,000 resamples, seed 42) that redraws the left-out humans and the val humans.

**Reading, declared:** the assumption **holds** for a fold if the paired difference's interval is below or contains zero (the ensemble's inflation is no larger than a single model's). It **fails** if the interval lies above zero (the ensemble flags more new-topic humans than a single model at the same $\tau$): then $\tau$ would be too lax for the ensemble and would have to be re-derived. Both folds are reported separately; two topics are a small basis, and I say so.

## 4 · Stability

All at the **final operating point** $\tau(1\%) = 0.00235$ (rule (d), notebook 18b); the band edge $\tau(5\%) = 0.01032$ is used for the typo test below.

**(i) Agreement of verdicts across seeds.** Among essays flagged by **at least one** of the three single seeds, the share flagged by **all three** (the statistic of 16b, where it was 28 of 72), on the val and test humans, the val and test AI essays, and the left-out humans of folds 7 and 11. For the ensemble the analogue uses the three **leave-one-seed-out sub-ensembles** (two seeds each). *Caveat declared in advance:* each pair of sub-ensembles shares a seed, which inflates their agreement; this measure therefore flatters the ensemble and is **an upper bound, not a fair comparison**. Pairwise Cohen's kappa is reported as well.

**(ii) Flip rate under 5% typos for essays near the threshold.** Each essay is scored again after `inject_typos(text, 0.05)` (two independent typo copies per essay). *Near the threshold* = the essay's own (unperturbed) $p$ lies within a **factor of 3 of $\tau(1\%)$** on either side ($\tau/3 \le p \le 3\tau$; for a single model $p$ is that model's, for the ensemble it is $p_{\text{ens}}$). A **flip** = the verdict (flagged / not flagged at $\tau(1\%)$) differs between the original essay and its typo copy. Reported: the flip rate among near-threshold essays and among all essays; the single models (average over the three seeds) against the ensemble; a paired bootstrap interval for their difference. Data: the val and test essays (the three saved final models, scored here) and the left-out essays of fold 11 (the three fold models, scored in `scripts/lopo_fold_typos.py`, because fold models are not kept). Fold 7 has no typo scores.

## 5 · The final system's operating points and zones

The final system is the ensemble of the three original-recipe seeds with the combination chosen in section 2, and the thresholds are the ones the declared choice rule picked for the original recipe (notebook 18b, rule (d), from all 13 topics): **$\tau(0.5\%) = 0.00093$, $\tau(1\%) = 0.00235$, $\tau(5\%) = 0.01032$**. Zones: **flagged** $p_{\text{ens}} \le \tau_{\text{flag}}$; **cannot tell** (to be worded as proposed in 18b, "unusual but common for student essays") $\tau_{\text{flag}} < p_{\text{ens}} \le \tau(5\%)$; **not flagged** above. Two operating points: **A**: $\tau_{\text{flag}} = \tau(1\%)$; **B**: $\tau_{\text{flag}} = \tau(0.5\%)$; the band's upper edge is $\tau(5\%)$ in both.

## 6 · The single final held-out check: the exact table (defined here, NOT computed)

**Rule of use.** Run **once**, at the end of this phase, for the **final ensemble + thresholds + band together**, after the project owner has seen this definition. No change to the ensemble, the combination, the thresholds or the band after it has been run. The held-out prompts ("Does the electoral college work?", "Summer projects") and the held-out generators (`darragh_claude_v6`, `darragh_claude_v7`, `falcon_180b_v1`) are scored by the three final models and combined exactly as in section 2 (the val-human calibration is the same; nothing is re-fitted).

**Table H1, human essays of each held-out prompt, separately** (rows: Electoral college; Summer projects; both pooled). For operating point A ($\tau_{\text{flag}} = \tau(1\%)$) and B ($\tau(0.5\%)$): *n*; **% flagged**, **% in the cannot-tell band** and **% not flagged**, each with a 95% Wilson interval (the pooled row: cluster bootstrap, 1,000 resamples, seed 42). Two blocks: all human essays, and on-topic human essays only.

**Table H2, AI essays of the held-out prompts** (same rows, same two operating points): **% flagged (AI caught)**, **% in the band**, **% missed**; all essays and on-topic only.

**Table H3, AI essays of the held-out generators** (rows: `darragh_claude_v6`, `darragh_claude_v7`, `falcon_180b_v1`, pooled): **% flagged**, **% in the band**, **% missed**, both operating points; all essays and on-topic only.

**Table H4, context (not part of the claims):** the same human false-flag and AI-caught quantities for the **single seed-42 model at the old logit thresholds of notebook 09** (4.67 for "1%", 7.80 for "0.5%"), for continuity with the numbers already in the README and model card.

**Table H5, the check against what Phase 4 predicted.** Registered now, from `results/18b_*` (original recipe, rule (d), leave-one-topic-out over 13 topics, single-seed models):

| quantity (operating point A, $\tau(1\%)$) | prediction |
|---|---|
| mean human false-flag rate over new topics | 0.58% (95% interval 0.44-0.88) |
| human false-flag rate of any one new topic | within the range of the 13 left-out topics: **0.00% to 1.74%** (worst topic); 23% of topics above 1% |
| AI essays caught, new topics | 98.78% (97.74-99.21) |
| share of human / AI essays in the cannot-tell band | 2.49% / 0.69% |

H5 reports, for each held-out prompt, the observed human false-flag rate against the 0.00-1.74% range and the AI caught against 97.7-99.2%, and says plainly where an observation falls outside what was predicted. Summer projects was the hardest unseen prompt for the single original model (7.7% at the old 1% threshold), so a value near or above the top of the range there would be a finding, not a surprise.

**Not part of the check and not computed:** anything that would let the held-out results feed back into a choice.

## 7 · What is declared but not part of this notebook's claims

- The single-seed folds are the basis of the Phase 4 thresholds; the ensemble check in section 3 rests on **two topics** and three seeds each.
- The leave-one-seed-out sub-ensembles in 4(i) share seeds; the typo test in 4(ii) is the cleaner stability evidence.
- Seeds 43 and 44 of the full-training models are trained here; seed 42 is the existing model. There is no ensemble of more than three.

In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.data import load_splits
from src.evaluate import wilson_interval, ClusterSampler
from src.conformal import tail_p

SEED, N_BOOT = 42, 1000
TAU = {'0.5%': 0.00093, '1%': 0.00235, '5%': 0.01032}      # notebook 18b, rule (d), from all 13 topics (declared in section 5)
pd.set_option('display.width', 230); pd.set_option('display.max_colwidth', 100)
data = load_splits()